In [ ]:
import math
import torch
import gpytorch
from matplotlib import pyplot as plt
import h5py
import numpy as np
import random

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.chdir("/lscratch/fgmaion/MTNG-resims/src")
import utils

In [ ]:
plt.rcParams["font.family"] = "serif"
plt.rcParams["mathtext.fontset"] = "dejavuserif"

##### Get the parameters

In [ ]:
omega_matter = []
sigma_8      = []

wind_en      = []
wind_vel     = []
radio_fb     = []
radio_fb_reo = []

with open("/scratch/fgmaion/CAMELS/LH/CosmoAstroSeed_IllustrisTNG_L25n256_LH.txt", 'r') as f:
    for line in f.readlines():
        try:
            omega_matter.append(float(line.split()[1]))
            sigma_8.append(float(line.split()[2]))
            wind_en.append(float(line.split()[3]))
            wind_vel.append(float(line.split()[5]))
            radio_fb.append(float(line.split()[4]))
            radio_fb_reo.append(float(line.split()[6]))

        except:
            continue

omega_matter = np.asarray(omega_matter)
sigma_8      = np.asarray(sigma_8)
wind_en      = np.asarray(wind_en)
wind_vel     = np.asarray(wind_vel)
radio_fb     = np.asarray(radio_fb)
radio_fb_reo = np.asarray(radio_fb_reo)

##### Get the baseline data

In [ ]:
base_data = utils.camels_stellar_mf('1P', baseline=True, nbins=15)

In [ ]:
def pars(i, mstar):

    arr = np.vstack( (mstar, np.ones(len(mstar)) * wind_en[i],\
                        np.ones(len(mstar)) * wind_vel[i],\
                        np.ones(len(mstar)) * radio_fb[i],\
                        np.ones(len(mstar)) * radio_fb_reo[i],\
                        np.ones(len(mstar)) * omega_matter[i],\
                        np.ones(len(mstar)) * sigma_8[i] )).T

    return arr

### Get the data

In [ ]:
smf = {}
for i in range(100):
    smf[i] = utils.camels_stellar_mf('LH', num=i, nbins=15)


In [ ]:
n_sets = [2,4,6,10,20,30,40,50,60]

train_sel = {}
test_sel = {}
for i in range(len(n_sets)):
    train_sel[i] = random.sample(range(100), n_sets[i])
    test_sel[i]  = random.sample(list(np.delete(range(100), train_sel[i])), 40)

### Build the GP Model

In [ ]:
# We will use the simplest form of GP model, exact inference
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(ExactGPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel(ard_num_dims=7))

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

import os
smoke_test = ('CI' in os.environ)
training_iter = 2 if smoke_test else 200

In [ ]:
model = {}
likelihood = {}
for n in range(len(n_sets)):

    #del smf_global, arr_global
    mstar = np.log10(smf[train_sel[n][0]]['mstar'][:-3])
    arr_global = pars(train_sel[n][0], mstar)
    smf_global = np.log10(smf[train_sel[n][0]]['smf'][:-3])

    for i in range(1,n_sets[n]):
        mstar = np.log10(smf[train_sel[n][i]]['mstar'][:-3])

        arr = pars(train_sel[n][i], mstar)

        arr_global = np.vstack((arr_global, arr))

        smf_global = np.hstack((smf_global, np.log10(smf[train_sel[n][i]]['smf'][:-3])))

    isnan = np.where(np.isnan(arr_global))[0]

    arr_global = np.delete(arr_global, isnan, axis=0)
    smf_global = np.delete(smf_global, isnan)

    train_x = torch.asarray(arr_global, dtype=torch.float)
    train_y = torch.asarray(smf_global, dtype=torch.float)

    # initialize likelihood and model
    likelihood[n] = gpytorch.likelihoods.GaussianLikelihood()
    model[n] = ExactGPModel(train_x, train_y, likelihood[n])

    # TRAIN THE MODEL

    # Find optimal model hyperparameters
    model[n].train()
    likelihood[n].train()

    # Use the adam optimizer
    optimizer = torch.optim.Adam(model[n].parameters(), lr=0.1)  # Includes GaussianLikelihood parameters

    # "Loss" for GPs - the marginal log likelihood
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood[n], model[n])

    for i in range(training_iter):
        # Zero gradients from previous iteration
        optimizer.zero_grad()
        # Output from model
        output = model[n](train_x)
        # Calc loss and backprop gradients
        loss = -torch.sum(mll(output, train_y))
        loss.backward()
        print('Iter %d/%d - Loss: %.3f   noise: %.3f' % (
            i + 1, training_iter, loss.item(),
            model[n].likelihood.noise.item()
        ))
        optimizer.step()

In [ ]:
mstar_test = np.linspace(8,11,15)
test_x1 = torch.asarray(pars(train_sel[5][5], np.log10(smf[train_sel[5][5]]['mstar'][:-5])), dtype=torch.float)
test_x2 = torch.asarray(pars(test_sel[5][10], np.log10(smf[test_sel[5][10]]['mstar'][:-5])), dtype=torch.float)

In [ ]:
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    print(observed_pred2.mean.numpy())

In [ ]:
diff = np.zeros((len(n_sets), len(smf[test_sel[5][0]]['smf'][:-6])))
for n in range(len(n_sets)):
    # Get into evaluation (predictive posterior) mode
    model[n].eval()
    likelihood[n].eval()

    # Test points are regularly spaced along [0,1]
    # Make predictions by feeding model through likelihood
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        observed_pred1 = likelihood[n](model[n](test_x1))
        observed_pred2 = likelihood[n](model[n](test_x2))

    for i in range(40):
        test_x2 = torch.asarray(pars(test_sel[5][i], np.log10(smf[test_sel[5][i]]['mstar'][:-6])), dtype=torch.float)
        observed_pred2 = likelihood[n](model[n](test_x2))

        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            diff[n] += (10**observed_pred2.mean.numpy() - smf[test_sel[5][i]]['smf'][:-6])**2 / 40

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_yscale('log')

for n in range(len(n_sets)):
    ax.plot(np.log10(smf[test_sel[5][i]]['mstar'][:-5]), diff[n])


In [ ]:
GAMA = np.array([
    [6.875, -0.691, 0.176],
    [7.125, -1.084, 0.125],
    [7.375, -1.011, 0.071],
    [7.625, -1.349, 0.092],
    [7.875, -1.287, 0.079],
    [8.125, -1.544, 0.071],
    [8.375, -1.669, 0.045],
    [8.625, -1.688, 0.032],
    [8.875, -1.795, 0.024],
    [9.125, -1.886, 0.020],
    [9.375, -2.055, 0.014],
    [9.625, -2.142, 0.010],
    [9.875, -2.219, 0.009],
    [10.125, -2.274, 0.009],
    [10.375, -2.292, 0.009],
    [10.625, -2.361, 0.010],
    [10.875, -2.561, 0.013],
    [11.125, -2.922, 0.019],
    [11.375, -3.414, 0.032],
    [11.625, -4.704, 0.138]
])

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

mstar = np.array([1.21913611e+08, 1.81306119e+08, 2.69955921e+08, 4.02197146e+08,
       5.98997204e+08, 8.91672492e+08, 1.32633065e+09, 1.97137091e+09,
       2.93026705e+09, 4.35252277e+09, 6.47292187e+09, 9.62527655e+09,
       1.43077758e+10, 2.12979515e+10, 3.17181189e+10, 4.71354545e+10,
       6.98670343e+10, 1.03530678e+11, 1.53281867e+11, 2.26870697e+11,
       3.37601338e+11, 5.04021002e+11, 7.49909506e+11, 1.11216571e+12,
       1.64311855e+12, 2.44407201e+12, 3.62656053e+12, 5.32495893e+12,
       8.02762401e+12])

err_smf = np.array([9.82910177e-04, 8.89547321e-04, 9.51583154e-04, 7.86680598e-04,
       8.97124019e-04, 8.01417120e-04, 7.14627717e-04, 6.51999056e-04,
       6.16794749e-04, 6.18095867e-04, 5.32409435e-04, 4.90570022e-04,
       4.45456922e-04, 3.52661420e-04, 2.95798619e-04, 2.74018815e-04,
       2.46385849e-04, 2.33045584e-04, 1.86210114e-04, 1.07758083e-04,
       7.81269504e-05, 5.57888721e-05, 4.68111596e-05, 2.50104609e-05,
       1.47976657e-05, 6.11589140e-06, 3.43450946e-06, 8.80813608e-07,
       1.29002626e-07])

smf_total = np.array([1.37203333e-02, 1.09983222e-02, 9.03347176e-03, 7.73941745e-03,
       6.93803608e-03, 6.40660873e-03, 6.00870888e-03, 5.59722262e-03,
       5.11285730e-03, 4.57682903e-03, 3.99812330e-03, 3.44979227e-03,
       2.94087911e-03, 2.53817643e-03, 2.23898713e-03, 1.99443189e-03,
       1.66843036e-03, 1.25439125e-03, 8.15645653e-04, 4.64562641e-04,
       2.54809941e-04, 1.43594712e-04, 8.29318593e-05, 4.55908882e-05,
       2.34228417e-05, 1.05575863e-05, 4.68745292e-06, 1.41344734e-06,
       4.47110894e-07])

ax.set_xscale('log')

for n in range(len(n_sets)):
    ax.plot(smf[test_sel[5][i]]['mstar'][:-6], np.log10(1+np.sqrt(diff[n]) / smf[test_sel[5][i]]['smf'][:-6]), label=n_sets[n])

ax.fill_between(10**GAMA[:,0], np.zeros(GAMA.shape[0]), GAMA[:,2], color='r', label='GAMA errors', alpha=0.1, edgecolor=None)
ax.fill_between(mstar, np.zeros(mstar.shape[0]), np.log10(1 + err_smf/smf_total), color='b', label='Reconstruction errors', alpha=0.1, edgecolor=None)

cv = np.array([0.02509286, 0.04012347, 0.03010014, 0.03498795, 0.03805   ,
       0.04930883, 0.05057655, 0.09816664])

ax.plot(smf[test_sel[5][i]]['mstar'][:-6], cv, color='k', label='CV')

ax.set_ylabel('dex')
ax.set_xlabel('$M_*$')

ax.legend(fontsize=6)

In [ ]:
with torch.no_grad():
    # Initialize plot
    f, ax = plt.subplots(1, 1, figsize=(5.5, 5), dpi=150)

    # Get upper and lower confidence bounds
    lower1, upper1 = observed_pred1.confidence_region()
    lower2, upper2 = observed_pred2.confidence_region()
    # Plot training data as black stars
    ax.plot(np.log10(smf[train_sel[5]]['mstar'][:-5]), np.log10(smf[train_sel[5]]['smf'][:-5]), 'b*', label='Train Set')
    ax.plot(np.log10(smf[test_sel[10]]['mstar'][:-5]), np.log10(smf[test_sel[10]]['smf'][:-5]), 'r*', label='Test Set')

    # Plot predictive means as blue line
    ax.plot(test_x1[:,0], observed_pred1.mean.numpy(), 'b')
    ax.plot(test_x2[:,0], observed_pred2.mean.numpy(), 'r')

    # Shade between the lower and upper confidence bounds
    ax.fill_between(test_x1.numpy()[:,0], observed_pred1.mean.numpy()-observed_pred1.stddev.numpy(), observed_pred1.mean.numpy()+observed_pred1.stddev.numpy(), color='b', alpha=0.5, edgecolor=None)
    ax.fill_between(test_x2.numpy()[:,0], observed_pred2.mean.numpy()-observed_pred2.stddev.numpy(), observed_pred2.mean.numpy()+observed_pred2.stddev.numpy(), color='r', alpha=0.5, edgecolor=None)

ax.legend()

ax.set_xlabel('$M_*$')
ax.set_ylabel('SMF')


In [ ]:
with torch.no_grad():
    # Initialize plot
    f, ax = plt.subplots(1, 2, figsize=(10, 5), dpi=150, sharey=True)
    
    plt.subplots_adjust(wspace=0, hspace=0)

    for i in range(40):
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            test_x1 = torch.asarray(pars(train_sel[i], np.log10(smf[train_sel[i]]['mstar'][:-5])), dtype=torch.float)
            observed_pred1 = likelihood(model(test_x1))
    
        ax[0].plot(np.log10(smf[train_sel[i]]['mstar'][:-5]),  np.log10(smf[train_sel[i]]['smf'][:-5]) - observed_pred1.mean.numpy(), color='b', lw=0.7)
        ax[0].fill_between(np.log10(smf[train_sel[i]]['mstar'][:-5]), np.log10(smf[train_sel[i]]['smf'][:-5]) - (observed_pred1.mean.numpy()-observed_pred1.stddev.numpy()) , np.log10(smf[train_sel[i]]['smf'][:-5]) - (observed_pred1.mean.numpy()+observed_pred1.stddev.numpy()), alpha=0.05, color='b', edgecolor=None)

    for i in range(60):
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            test_x2 = torch.asarray(pars(test_sel[i], np.log10(smf[test_sel[i]]['mstar'][:-5])), dtype=torch.float)
            observed_pred2 = likelihood(model(test_x2))
        
        ax[1].plot(np.log10(smf[test_sel[i]]['mstar'][:-5]), np.log10(smf[test_sel[i]]['smf'][:-5]) - observed_pred2.mean.numpy(), color='r', lw=0.7)
        ax[1].fill_between(np.log10(smf[test_sel[i]]['mstar'][:-5]), np.log10(smf[test_sel[i]]['smf'][:-5]) - (observed_pred2.mean.numpy()-observed_pred2.stddev.numpy()) , np.log10(smf[test_sel[i]]['smf'][:-5]) - (observed_pred2.mean.numpy()+observed_pred1.stddev.numpy()), alpha=0.05, color='r', edgecolor=None)
#        ax[1].fill_between(np.log10(smf[test_sel[i]]['mstar'][:-5]), np.log10(smf[test_sel[i]]['smf'][:-5]) - lower2.numpy(), np.log10(smf[test_sel[i]]['smf'][:-5]) - upper2.numpy(), alpha=0.01, color='r', edgecolor=None)

    # Plot training data as black stars
    ax[0].axhline(0, color='k', ls='--')
    ax[1].axhline(0, color='k', ls='--')

ax[0].set_ylabel('$\Delta \log_{10}(\\mathrm{SMF})$')
ax[0].set_xlabel('Stellar Mass $M_*[M_\odot]$')
ax[1].set_xlabel('Stellar Mass $M_*[M_\odot]$')

ax[0].set_title('Training Data [1-40]')
ax[1].set_title('Validation Data [41-100]')

In [ ]:
with torch.no_grad():
    # Initialize plot
    f, ax = plt.subplots(1, 2, figsize=(10, 5), dpi=150, sharey=True)
    
    plt.subplots_adjust(wspace=0, hspace=0)

    # Get upper and lower confidence bounds
    lower1, upper1 = observed_pred1.confidence_region()
    lower2, upper2 = observed_pred2.confidence_region()
    # Plot training data as black stars
    ax[0].axhline(1, color='k', ls='--')
    ax[1].axhline(1, color='k', ls='--')

    for i in range(40):
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            test_x1 = torch.asarray(pars(train_sel[i], np.log10(smf[train_sel[i]]['mstar'][:-5])), dtype=torch.float)
            observed_pred1 = likelihood(model(test_x1))
            lower1, upper1 = observed_pred1.confidence_region()
    
        ax[0].plot(np.log10(smf[train_sel[i]]['mstar'][:-5]),  smf[train_sel[i]]['smf'][:-5] / 10**observed_pred1.mean.numpy(), color='b', ls='', marker='o', ms=1)

    for i in range(60):
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            test_x2 = torch.asarray(pars(test_sel[i], np.log10(smf[test_sel[i]]['mstar'][:-4])), dtype=torch.float)
            observed_pred2 = likelihood(model(test_x2))
            lower2, upper2 = observed_pred2.confidence_region()
        
        ax[1].plot(np.log10(smf[test_sel[i]]['mstar'][:-4]), smf[test_sel[i]]['smf'][:-4] / 10**observed_pred2.mean.numpy(), color='r', ls='', marker='o', ms=1)

ax[0].set_ylabel('SMF Ratio')
ax[0].set_xlabel('Stellar Mass $M_*[M_\odot]$')
ax[1].set_xlabel('Stellar Mass $M_*[M_\odot]$')

ax[0].set_title('Training Data [1-40]')
ax[1].set_title('Validation Data [41-100]')